In [ ]:
!pip install -q soundfile

import torch, torchaudio
print(f'PyTorch {torch.__version__} | torchaudio {torchaudio.__version__}')

if torch.cuda.is_available():
    device = 'cuda'
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {torch.cuda.get_device_name(0)} ({props.total_memory/1e9:.1f} GB)')
else:
    device = 'cpu'
    print('WARNING: no GPU — training will be very slow')
print('device:', device)


In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

CKPT_DIR = '/content/drive/MyDrive/vocoder_gp2'
os.makedirs(CKPT_DIR, exist_ok=True)
print('Checkpoints will be saved to:', CKPT_DIR)


In [ ]:
import os
DATA_DIR = '/content/LJSpeech-1.1'

if not os.path.isdir(DATA_DIR):
    !wget -q --show-progress -O /content/lj.tar.bz2 \
        https://data.keithito.com/data/speech/LJSpeech-1.1.tar.bz2
    !tar -xjf /content/lj.tar.bz2 -C /content && rm /content/lj.tar.bz2

print(f'LJSpeech: {len(os.listdir(DATA_DIR+"/wavs"))} files at {DATA_DIR}')


In [ ]:
import json, os, random
from dataclasses import dataclass, field
from pathlib import Path
from typing import List

import numpy as np
import soundfile as sf
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm
from torch.nn.utils import spectral_norm
from torch.nn.utils.parametrizations import weight_norm
from torch.nn.utils.parametrize import remove_parametrizations

@dataclass
class Config:
    # пути
    data_path:      str = '/content/LJSpeech-1.1'
    mel_cache:      str = '/content/mel_cache'
    checkpoint_dir: str = CKPT_DIR

    sample_rate:    int   = 22050
    n_fft:          int   = 1024
    win_length:     int   = 1024
    hop_length:     int   = 256
    num_mels:       int   = 80
    mel_fmin:       float = 0.0
    mel_fmax:       float = 8000.0
    mel_power:      float = 1.5
    # данные
    segment_size:   int = 8192
    batch_size:     int = 16
    num_workers:    int = 2

    upsample_rates:           List[int]       = field(default_factory=lambda: [8,8,2,2])
    upsample_kernel_sizes:    List[int]       = field(default_factory=lambda: [16,16,4,4])
    upsample_initial_channel: int             = 128
    resblock_kernel_sizes:    List[int]       = field(default_factory=lambda: [3,7,11])
    resblock_dilation_sizes:  List[List[int]] = field(
        default_factory=lambda: [[1,3,5],[1,3,5],[1,3,5]])
    # обучение
    lr:          float = 2e-4
    adam_b1:     float = 0.8
    adam_b2:     float = 0.99
    lr_decay:    float = 0.999
    num_epochs:  int   = 300
    save_every:  int   = 10
    lambda_mel:  float = 45.0
    lambda_fm:   float = 2.0

cfg = Config()
print('Config OK | batch=%d | epochs=%d | lr=%.0e' % (cfg.batch_size, cfg.num_epochs, cfg.lr))


In [ ]:
def mel_from_audio(audio: torch.Tensor, cfg: Config) -> torch.Tensor:
    """Log-mel spectrogram matching FastPitch TTS audio config."""
    audio = audio.cpu().float()
    win = torch.hann_window(cfg.win_length)
    spec = torch.stft(audio, n_fft=cfg.n_fft, hop_length=cfg.hop_length,
                      win_length=cfg.win_length, window=win,
                      return_complex=True).abs()              # [F, T]
    fb = torchaudio.functional.melscale_fbanks(
        n_freqs=cfg.n_fft//2+1, f_min=cfg.mel_fmin, f_max=cfg.mel_fmax,
        n_mels=cfg.num_mels, sample_rate=cfg.sample_rate,
        norm=None, mel_scale='htk')                           # [F, M]
    mel = torch.matmul(spec.T, fb).T                          # [M, T]
    return torch.log(torch.clamp(mel ** cfg.mel_power, min=1e-5))


class LJSpeechDataset(Dataset):
    def __init__(self, cfg: Config, split='train'):
        self.cfg  = cfg
        self.seg  = cfg.segment_size
        self.mseg = cfg.segment_size // cfg.hop_length
        self.cache = Path(cfg.mel_cache)

        files = sorted((Path(cfg.data_path)/'wavs').glob('*.wav'))
        assert files, 'No wav files found'
        random.Random(42).shuffle(files)
        cut = min(200, int(0.05*len(files)))
        self.files = files[cut:] if split=='train' else files[:cut]
        self._build_cache()

    def _build_cache(self):
        self.cache.mkdir(parents=True, exist_ok=True)
        miss = [p for p in self.files if not (self.cache/(p.stem+'.npy')).exists()]
        if not miss: return
        print(f'Building mel cache ({len(miss)} files)...')
        for p in tqdm(miss, leave=False):
            data, sr = sf.read(str(p), dtype='float32', always_2d=False)
            audio = torch.from_numpy(data)
            if audio.dim()==2: audio = audio.mean(1)
            if sr != self.cfg.sample_rate:
                audio = torchaudio.functional.resample(audio, sr, self.cfg.sample_rate)
            np.save(str(self.cache/(p.stem+'.npy')), mel_from_audio(audio, self.cfg).numpy())

    def __len__(self): return len(self.files)

    def __getitem__(self, idx):
        p = self.files[idx]
        mel = torch.from_numpy(np.load(str(self.cache/(p.stem+'.npy'))))  # [M, T]
        T = mel.shape[1]
        t0 = random.randint(0, max(0, T-self.mseg))
        mel = mel[:, t0:t0+self.mseg]
        if mel.shape[1] < self.mseg:
            mel = F.pad(mel, (0, self.mseg-mel.shape[1]))

        a0 = t0 * self.cfg.hop_length
        data, sr = sf.read(str(p), dtype='float32', always_2d=False,
                           start=a0, frames=self.seg)
        audio = torch.from_numpy(data)
        if audio.dim()==2: audio = audio.mean(1)
        if sr != self.cfg.sample_rate:
            audio = torchaudio.functional.resample(audio, sr, self.cfg.sample_rate)
        if audio.shape[0] < self.seg:
            audio = F.pad(audio, (0, self.seg-audio.shape[0]))
        return mel, audio[:self.seg]


print('Building train dataset (mel cache on first run takes ~3 min)...')
train_ds = LJSpeechDataset(cfg, split='train')
print(f'Train: {len(train_ds)} files')


In [ ]:
# Генератор + мульти-периодный + мульти-масштабный дискриминаторы
LRELU = 0.1

class ResBlock(nn.Module):
    def __init__(self, ch, k, dil):
        super().__init__()
        self.c1 = nn.ModuleList([weight_norm(
            nn.Conv1d(ch,ch,k,dilation=d,padding=(k*d-d)//2)) for d in dil])
        self.c2 = nn.ModuleList([weight_norm(
            nn.Conv1d(ch,ch,k,padding=(k-1)//2)) for _ in dil])
    def forward(self,x):
        for a,b in zip(self.c1,self.c2):
            x = x + b(F.leaky_relu(a(F.leaky_relu(x,LRELU)),LRELU))
        return x
    def remove_wn(self):
        for c in list(self.c1)+list(self.c2): remove_parametrizations(c,'weight')


class Generator(nn.Module):
    def __init__(self,cfg):
        super().__init__()
        self.nk=len(cfg.resblock_kernel_sizes)
        ch=cfg.upsample_initial_channel
        self.pre=weight_norm(nn.Conv1d(cfg.num_mels,ch,7,padding=3))
        self.ups,self.res=nn.ModuleList(),nn.ModuleList()
        for i,(u,k) in enumerate(zip(cfg.upsample_rates,cfg.upsample_kernel_sizes)):
            och=ch>>(i+1)
            self.ups.append(weight_norm(nn.ConvTranspose1d(ch>>i,och,k,stride=u,padding=(k-u)//2)))
            for kr,dr in zip(cfg.resblock_kernel_sizes,cfg.resblock_dilation_sizes):
                self.res.append(ResBlock(och,kr,dr))
        self.post=weight_norm(nn.Conv1d(och,1,7,padding=3))
    def forward(self,x):
        x=self.pre(x)
        for i,up in enumerate(self.ups):
            x=up(F.leaky_relu(x,LRELU))
            x=sum(self.res[i*self.nk+j](x) for j in range(self.nk))/self.nk
        return torch.tanh(self.post(F.leaky_relu(x,LRELU)))
    def remove_weight_norm(self):
        remove_parametrizations(self.pre,'weight')
        [remove_parametrizations(u,'weight') for u in self.ups]
        [r.remove_wn() for r in self.res]
        remove_parametrizations(self.post,'weight')


class PeriodDisc(nn.Module):
    def __init__(self,p):
        super().__init__()
        self.p=p
        ch=[1,32,128,512,1024,1024]
        self.convs=nn.ModuleList([weight_norm(
            nn.Conv2d(ch[i],ch[i+1],(5,1),(3,1),padding=(2,0))) for i in range(5)])
        self.post=weight_norm(nn.Conv2d(1024,1,(3,1),padding=(1,0)))
    def forward(self,x):
        b,c,t=x.shape
        if t%self.p: x=F.pad(x,(0,self.p-t%self.p),'reflect'); t=x.shape[-1]
        x=x.view(b,c,t//self.p,self.p)
        fmap=[]
        for conv in self.convs: x=F.leaky_relu(conv(x),LRELU); fmap.append(x)
        x=self.post(x); fmap.append(x)
        return x.flatten(1,-1),fmap


class MPD(nn.Module):
    def __init__(self): super().__init__(); self.d=nn.ModuleList([PeriodDisc(p) for p in [2,3,5,7,11]])
    def forward(self,yr,yf):
        ro,fo,rf,ff=[],[],[],[]
        for d in self.d:
            r,rm=d(yr); f,fm=d(yf)
            ro.append(r);fo.append(f);rf.append(rm);ff.append(fm)
        return ro,fo,rf,ff


class ScaleDisc(nn.Module):
    def __init__(self,sn=False):
        super().__init__()
        n=spectral_norm if sn else weight_norm
        self.convs=nn.ModuleList([
            n(nn.Conv1d(1,   128,15,1,padding=7)),
            n(nn.Conv1d(128, 128,41,2,groups=4, padding=20)),
            n(nn.Conv1d(128, 256,41,2,groups=16,padding=20)),
            n(nn.Conv1d(256, 512,41,4,groups=16,padding=20)),
            n(nn.Conv1d(512,1024,41,4,groups=16,padding=20)),
            n(nn.Conv1d(1024,1024,41,1,groups=16,padding=20)),
            n(nn.Conv1d(1024,1024,5,1,padding=2)),
        ])
        self.post=n(nn.Conv1d(1024,1,3,1,padding=1))
    def forward(self,x):
        fmap=[]
        for c in self.convs: x=F.leaky_relu(c(x),LRELU); fmap.append(x)
        x=self.post(x); fmap.append(x)
        return x.flatten(1,-1),fmap


class MSD(nn.Module):
    def __init__(self):
        super().__init__()
        self.d=nn.ModuleList([ScaleDisc(sn=True),ScaleDisc(),ScaleDisc()])
        self.pool=nn.ModuleList([nn.AvgPool1d(4,2,padding=2),nn.AvgPool1d(4,2,padding=2)])
    def forward(self,yr,yf):
        ro,fo,rf,ff=[],[],[],[]
        for i,d in enumerate(self.d):
            if i: yr=self.pool[i-1](yr); yf=self.pool[i-1](yf)
            r,rm=d(yr); f,fm=d(yf)
            ro.append(r);fo.append(f);rf.append(rm);ff.append(fm)
        return ro,fo,rf,ff


G   = Generator(cfg).to(device)
mpd = MPD().to(device)
msd = MSD().to(device)
print(f'Generator: {sum(p.numel() for p in G.parameters())/1e6:.2f}M params')


In [ ]:
# потери: адверсарная + feature matching + mel-L1

def loss_disc(ro, fo):
    return sum(torch.mean((r-1)**2) + torch.mean(f**2) for r,f in zip(ro,fo))

def loss_gen_adv(fo):
    return sum(torch.mean((f-1)**2) for f in fo)

def loss_feat_match(rf, ff):
    return sum(F.l1_loss(f,r.detach()) for rl,fl in zip(rf,ff) for r,f in zip(rl,fl))

# torch.stft не поддерживается на MPS — считаем на CPU
_win = torch.hann_window(cfg.win_length)
_fb  = torchaudio.functional.melscale_fbanks(
    n_freqs=cfg.n_fft//2+1, f_min=cfg.mel_fmin, f_max=cfg.mel_fmax,
    n_mels=cfg.num_mels, sample_rate=cfg.sample_rate, norm=None, mel_scale='htk')

def loss_mel(yr, yf):
    def to_mel(x):
        a = x.squeeze(1).cpu()
        s = torch.stft(a.reshape(-1,a.shape[-1]), n_fft=cfg.n_fft,
                       hop_length=cfg.hop_length, win_length=cfg.win_length,
                       window=_win, return_complex=True).abs()
        m = torch.matmul(s.permute(0,2,1),_fb).permute(0,2,1)
        return torch.log(torch.clamp(m**cfg.mel_power, min=1e-5))
    dev = yr.device
    return F.l1_loss(to_mel(yf).to(dev), to_mel(yr).to(dev))

print('Losses defined: adversarial (LS-GAN) + feature matching + mel-L1')


In [ ]:
os.makedirs(cfg.checkpoint_dir, exist_ok=True)

loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                    num_workers=cfg.num_workers, pin_memory=True, drop_last=True)

og = torch.optim.AdamW(G.parameters(), lr=cfg.lr, betas=(cfg.adam_b1,cfg.adam_b2))
od = torch.optim.AdamW(list(mpd.parameters())+list(msd.parameters()),
                        lr=cfg.lr, betas=(cfg.adam_b1,cfg.adam_b2))
sg = torch.optim.lr_scheduler.ExponentialLR(og, cfg.lr_decay)
sd = torch.optim.lr_scheduler.ExponentialLR(od, cfg.lr_decay)

RESUME = None
start_epoch, step = 0, 0
if RESUME and os.path.exists(RESUME):
    c = torch.load(RESUME, map_location=device)
    G.load_state_dict(c['G']); mpd.load_state_dict(c['MPD']); msd.load_state_dict(c['MSD'])
    og.load_state_dict(c['og']); od.load_state_dict(c['od'])
    start_epoch, step = c['epoch'], c['step']
    print('Resumed from epoch', start_epoch)

metrics_path = f'{cfg.checkpoint_dir}/metrics.json'
history = {'epoch': [], 'd_loss': [], 'g_loss': [], 'adv_loss': [], 'fm_loss': [], 'mel_loss': [], 'lr': []}
if RESUME and os.path.exists(metrics_path):
    with open(metrics_path) as _f:
        history = json.load(_f)

print(f'Train: {len(train_ds)} samples | {len(loader)} batches/epoch')

epoch_bar = tqdm(range(start_epoch, cfg.num_epochs), desc='Epochs')
for epoch in epoch_bar:
    G.train(); mpd.train(); msd.train()
    bb = tqdm(loader, desc=f'ep{epoch+1}', leave=False)

    sum_d, sum_g, sum_adv, sum_fm, sum_ml, n_b = 0., 0., 0., 0., 0., 0

    for mel, audio in bb:
        mel  = mel.to(device)
        real = audio.unsqueeze(1).to(device)
        fake = G(mel)
        L = min(real.shape[-1], fake.shape[-1])
        real, fake = real[...,:L], fake[...,:L]

        # шаг дискриминатора
        od.zero_grad()
        ro1,fo1,_,_ = mpd(real,fake.detach())
        ro2,fo2,_,_ = msd(real,fake.detach())
        d = loss_disc(ro1,fo1) + loss_disc(ro2,fo2)
        d.backward(); od.step()

        # шаг генератора
        og.zero_grad()
        ro1,fo1,rf1,ff1 = mpd(real,fake)
        ro2,fo2,rf2,ff2 = msd(real,fake)
        adv  = loss_gen_adv(fo1) + loss_gen_adv(fo2)
        fm   = cfg.lambda_fm  * (loss_feat_match(rf1,ff1) + loss_feat_match(rf2,ff2))
        ml   = cfg.lambda_mel * loss_mel(real,fake)
        g = adv + fm + ml
        g.backward(); og.step()
        step += 1

        sum_d += d.item(); sum_g += g.item()
        sum_adv += adv.item(); sum_fm += fm.item(); sum_ml += ml.item()
        n_b += 1

        bb.set_postfix(D=f'{d.item():.3f}', G=f'{g.item():.3f}', mel=f'{ml.item():.2f}')

    sg.step(); sd.step()
    lr_now = og.param_groups[0]['lr']

    avg_d, avg_g = sum_d/n_b, sum_g/n_b
    avg_adv, avg_fm, avg_ml = sum_adv/n_b, sum_fm/n_b, sum_ml/n_b

    history['epoch'].append(epoch+1)
    history['d_loss'].append(round(avg_d, 5))
    history['g_loss'].append(round(avg_g, 5))
    history['adv_loss'].append(round(avg_adv, 5))
    history['fm_loss'].append(round(avg_fm, 5))
    history['mel_loss'].append(round(avg_ml, 5))
    history['lr'].append(lr_now)
    with open(metrics_path, 'w') as _f:
        json.dump(history, _f, indent=2)

    epoch_bar.set_postfix(D=f'{avg_d:.3f}', G=f'{avg_g:.3f}', mel=f'{avg_ml:.3f}', lr=f'{lr_now:.2e}')

    ckpt_path = f'{cfg.checkpoint_dir}/epoch{epoch+1:04d}.pt'
    torch.save({'G':G.state_dict(),'MPD':mpd.state_dict(),'MSD':msd.state_dict(),
                'og':og.state_dict(),'od':od.state_dict(),
                'epoch':epoch+1,'step':step}, ckpt_path)

FINAL_CKPT = f'{cfg.checkpoint_dir}/vocoder_final.pt'
torch.save({'G':G.state_dict(),'MPD':mpd.state_dict(),'MSD':msd.state_dict(),
            'og':og.state_dict(),'od':od.state_dict(),
            'epoch':cfg.num_epochs,'step':step}, FINAL_CKPT)
print('Training complete. Final checkpoint:', FINAL_CKPT)


In [ ]:
import matplotlib.pyplot as plt
import json

with open(f'{cfg.checkpoint_dir}/metrics.json') as f:
    history = json.load(f)

epochs = history['epoch']

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].set_title('Generator losses')
for key, label in [('g_loss','G total'),('adv_loss','Adversarial'),('fm_loss','Feature matching'),('mel_loss','Mel')]:
    axes[0].plot(epochs, history[key], label=label)
axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].set_title('Discriminator loss')
axes[1].plot(epochs, history['d_loss'], label='D total', color='tab:red')
axes[1].set_xlabel('Epoch'); axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].set_title('Learning rate')
axes[2].plot(epochs, history['lr'], color='tab:green')
axes[2].set_xlabel('Epoch'); axes[2].set_yscale('log'); axes[2].grid(alpha=0.3)

plt.tight_layout()
plot_path = f'{cfg.checkpoint_dir}/training_curves.png'
plt.savefig(plot_path, dpi=130)
plt.show()
print('График сохранён:', plot_path)


In [ ]:
import os, IPython.display as ipd

# загружаем обученный вокодер
CKPT = FINAL_CKPT

G_inf = Generator(cfg).to(device)
G_inf.load_state_dict(torch.load(CKPT, map_location=device)['G'])
G_inf.eval()
G_inf.remove_weight_norm()
print('Vocoder loaded:', CKPT)

# Tacotron2 для text->mel (уже в torchaudio)
bundle   = torchaudio.pipelines.TACOTRON2_WAVERNN_CHAR_LJSPEECH
tac2     = bundle.get_tacotron2().to(device)
textproc = bundle.get_text_processor()
print('Tacotron2 loaded (LJSpeech, 22050 Hz, 80 mels)')

def text_to_mel(text: str) -> np.ndarray:
    with torch.inference_mode():
        tokens, lengths = textproc(text)
        mel, mel_len, _ = tac2.infer(tokens.to(device), lengths.to(device))
    mel = mel[0, :, :mel_len[0]].cpu()   # [80, T]
    # Tacotron2 даёт log(mel), вокодер ждёт log(mel^1.5) = 1.5*log(mel)
    return (mel * cfg.mel_power).numpy()

TEST_SENTENCES = [
    'Tourists found the octagonal lighthouse after a two-mile hike through fog',
    'In twenty forty-nine, neural implants became standard for city workers',
    'He whispered something in Icelandic I could not quite translate',
    'Lucia balanced a porcelain vase on her elbow without flinching',
    'They gathered quietly beneath the turbine as the wind picked up speed',
]

SAMPLES_DIR = '/content/samples'
os.makedirs(SAMPLES_DIR, exist_ok=True)

for i, text in enumerate(TEST_SENTENCES):
    print(f'\n[{i+1}/5] {text}')
    mel_t = torch.tensor(text_to_mel(text)).unsqueeze(0).to(device)  # [1, 80, T]
    with torch.inference_mode():
        wav = G_inf(mel_t).squeeze().cpu()
    path = f'{SAMPLES_DIR}/sample_{i+1:02d}.wav'
    torchaudio.save(path, wav.unsqueeze(0), cfg.sample_rate)
    print(f'  saved: {path}')
    ipd.display(ipd.Audio(wav.numpy(), rate=cfg.sample_rate))

# копируем на Drive
import shutil
dst = CKPT_DIR + '/samples'
os.makedirs(dst, exist_ok=True)
for fn in os.listdir(SAMPLES_DIR):
    shutil.copy(f'{SAMPLES_DIR}/{fn}', f'{dst}/{fn}')
print('\nAll samples saved to Drive:', dst)


In [ ]:

import os, shutil
import numpy as np
import torch, torchaudio
import IPython.display as ipd

CKPT = '/content/drive/MyDrive/vocoder_gp2/vocoder_final.pt'

G_inf = Generator(cfg).to(device)
G_inf.load_state_dict(torch.load(CKPT, map_location=device)['G'])
G_inf.eval()
G_inf.remove_weight_norm()
print('Vocoder loaded from:', CKPT)

bundle   = torchaudio.pipelines.TACOTRON2_WAVERNN_CHAR_LJSPEECH
tac2     = bundle.get_tacotron2().to(device)
textproc = bundle.get_text_processor()

def text_to_mel(text):
    with torch.inference_mode():
        tokens, lengths = textproc(text)
        mel, mel_len, _ = tac2.infer(tokens.to(device), lengths.to(device))
    mel = mel[0, :, :mel_len[0]].cpu()
    return (mel * cfg.mel_power).numpy()

# ── Тестовые фразы ───────────────────────────────────────────────────
TEST_SENTENCES = [
    'Tourists found the octagonal lighthouse after a two-mile hike through fog',
    'In twenty forty-nine, neural implants became standard for city workers',
    'He whispered something in Icelandic I could not quite translate',
    'Lucia balanced a porcelain vase on her elbow without flinching',
    'They gathered quietly beneath the turbine as the wind picked up speed',
]

SAMPLES_DIR = '/content/samples'
os.makedirs(SAMPLES_DIR, exist_ok=True)

for i, text in enumerate(TEST_SENTENCES):
    print(f'[{i+1}/5] {text}')
    mel_t = torch.tensor(text_to_mel(text)).unsqueeze(0).to(device)
    with torch.inference_mode():
        wav = G_inf(mel_t).squeeze().cpu()
    path = f'{SAMPLES_DIR}/sample_{i+1:02d}.wav'
    torchaudio.save(path, wav.unsqueeze(0), cfg.sample_rate)
    print(f'  -> {path}')
    ipd.display(ipd.Audio(wav.numpy(), rate=cfg.sample_rate))

# сохраняем на Drive
drive_samples = '/content/drive/MyDrive/vocoder_gp2/samples'
os.makedirs(drive_samples, exist_ok=True)
for fn in os.listdir(SAMPLES_DIR):
    shutil.copy(f'{SAMPLES_DIR}/{fn}', f'{drive_samples}/{fn}')
print(f'\nСемплы сохранены на Drive: {drive_samples}')

from google.colab import files
for fn in sorted(os.listdir(SAMPLES_DIR)):
    files.download(f'{SAMPLES_DIR}/{fn}')
